# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

> **Finding I'm auditing #1 — ML Appendix: "What Predicts Health? (Random Forest)"**
>
> The paper reports Average Position (43%), Impressions (32%), and Scroll Depth (15%) as the top predictors of Health
> Score, with CTR at 8% and Clicks at 2%. To their credit, the chart itself discloses the core limitation up front: *"the
target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal."* The
> body text repeats this — the ranking shows "which features the model uses most," not an external causal claim.
>
> **My methodology question, given that disclosure:** since the paper already flags the top-line
> position/impressions/scroll-depth importance as expected and non-causal, the more interesting question is about the
*remaining* features. Health Score's formula weights CTR at 20 of 100 points — the same order of magnitude as Scroll
> Depth's 20 points — yet CTR's importance (8%) is roughly half of Scroll Depth's (15%), and far below Position's (43%) or
> Impressions' (32%), which are each also worth only 30 points in the formula. If the label weights are roughly
> comparable, why is the model's reliance on them so uneven? One explanation: Position and Impressions may be acting as
*proxies* for CTR too, if pages with better position also tend to have better CTR in this dataset — the model could be
> reading CTR's signal indirectly through a correlated feature rather than through CTR itself. That's a different (and
> more subtle) leakage story than "the label leaks into the features" — it's "some features leak into each other," and it
> would make the importance ranking even less interpretable as a feature-by-feature breakdown. I'd ask the authors whether
> pairwise correlation among the four label-components was checked before reporting this chart.

> **Finding I'm auditing #2 — Finding #4: "The Freshness Multiplier"**
>
> The paper reports a 361+ day freshness window shows a 283:1 growth-to-decline ratio, and flags it themselves as "
> statistically unstable sample" — honest framing on their part.
>
> **My methodology question:** even setting sample-size instability aside, is there a selection effect in *which* pages
> get refreshed? Pages aren't refreshed at random — a team is more likely to refresh a page it already believes is worth
> saving (e.g., one still pulling meaningful impressions, or one with an obvious, low-effort fix). If refreshed pages were
> pre-selected for recoverability, the observed "freshness effect" partly reflects *which pages were chosen*, not what
> refreshing *does* to an arbitrary page. I'd ask whether the validation design includes a comparison group of
> similarly-old, non-refreshed pages, or whether this is a pure before/after aggregate on the refreshed set alone.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier

# --- Reload the same data + label + features as Week 5 ---
file_path = "../../data/raw/content_refresh_anonymized.csv"
if not os.path.exists(file_path):
    file_path = "https://raw.githubusercontent.com/Bibek-Dhakal/applied-search-intelligence/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(file_path)
df = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)].copy()
df = df.reset_index(drop=True)  # <-- keeps iloc/index aligned for every split below

peer_median = df.groupby(["position_tier", "content_type"])["ctr"].transform("median")
df["is_ctr_anomaly"] = (df["ctr"] < 0.5 * peer_median).astype(int)

FEATURES = ["impressions_90d", "avg_position", "content_age_days", "word_count", "engagement_rate"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

X = df[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_ctr_anomaly"].values

# --- BEFORE: naive random row split (what a first-pass eval often looks like) ---
X_tr_naive, X_te_naive, y_tr_naive, y_te_naive = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
rf_naive = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight="balanced", random_state=42, n_jobs=-1)
rf_naive.fit(X_tr_naive, y_tr_naive)
scores_naive = rf_naive.predict_proba(X_te_naive)[:, 1]

# --- AFTER: grouped, client-holdout split (my honest Week 5 design) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
X_tr_grp, X_te_grp = X.iloc[train_idx], X.iloc[test_idx]
y_tr_grp, y_te_grp = y[train_idx], y[test_idx]

rf_grp = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight="balanced", random_state=42, n_jobs=-1)
rf_grp.fit(X_tr_grp, y_tr_grp)
scores_grp = rf_grp.predict_proba(X_te_grp)[:, 1]

print("=== BEFORE vs AFTER: same model, same features, different split ===\n")
for k in (20, 50):
    p_naive = precision_at_k(scores_naive, y_te_naive, k)
    p_grp = precision_at_k(scores_grp, y_te_grp, k)
    print(f"Precision@{k}  |  naive random split: {p_naive:.3f}   |   client-grouped split: {p_grp:.3f}   |   gap: {p_naive - p_grp:+.3f}")

# Check for client overlap in the naive split — this is the thing that inflates it
naive_test_client_overlap = (
    set(df.loc[X_tr_naive.index, "client_id"])
    & set(df.loc[X_te_naive.index, "client_id"])
)
print(f"\nClients appearing in BOTH train and test under naive split: {len(naive_test_client_overlap)} of {df['client_id'].nunique()} total")

=== BEFORE vs AFTER: same model, same features, different split ===

Precision@20  |  naive random split: 1.000   |   client-grouped split: 0.850   |   gap: +0.150
Precision@50  |  naive random split: 0.940   |   client-grouped split: 0.640   |   gap: +0.300

Clients appearing in BOTH train and test under naive split: 26 of 30 total


> **Before/after interpretation:** The gap is not small — it's the headline finding of this audit. At Precision@20, the
> naive random split scores a perfect **1.000**, while the honest client-grouped split drops to **0.850** (a 0.150 gap).
> At Precision@50 the gap widens sharply: **0.940** naive vs **0.640** grouped — a **0.300** drop. The client-overlap
> check explains why: **26 of 30 clients (87%)** appear in both train and test under the naive split. With that much
> overlap, the model had ample opportunity to memorize client-specific CTR baselines rather than learn a generalizable
> pattern.
>
> This is directionally the same failure mode the paper's own Health Score model showed under stricter validation (
> 0.996 → 0.496 in the video's walkthrough) — a naive split that looks almost perfect collapses once client identity is
> properly held out. The honest number here, **Precision@50 = 0.640**, is the one that should be reported and is
> consistent with what Week 5 already found under the same grouped design. The takeaway: **the naive 0.940–1.000 figures
were never real** — they were measuring how well the model remembered which client a page belonged to, not whether it
> could rank CTR anomalies on unseen content.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# --- Attack my own Week-5 feature set the way the video described ---

# 1. Label-derived check: is CTR (or anything CTR-adjacent) hiding in my features?
print("Features in use:", FEATURES)
print("Confirmed excluded: 'ctr', 'clicks_90d' — both are directly used to build the label 'is_ctr_anomaly'\n")

# 2. Train WITH a deliberately reintroduced leaky feature to confirm the harness itself can detect leakage
X_leak_check = df[FEATURES + ["ctr"]].replace([np.inf, -np.inf], np.nan).fillna(0)
rf_leak = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight="balanced", random_state=42, n_jobs=-1)

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx2, te_idx2 = next(gss2.split(df, groups=df["client_id"]))
rf_leak.fit(X_leak_check.iloc[tr_idx2], y[tr_idx2])
leak_scores = rf_leak.predict_proba(X_leak_check.iloc[te_idx2])[:, 1]

print("=== Leakage sanity test: reintroducing 'ctr' as a feature ===")
for k in (20, 50):
    clean_p = precision_at_k(scores_grp, y_te_grp, k)
    leak_p = precision_at_k(leak_scores, y[te_idx2], k)
    print(f"Precision@{k}  |  honest features: {clean_p:.3f}   |   with 'ctr' reintroduced: {leak_p:.3f}")

# Feature importance with the leak present — expect 'ctr' to dominate, confirming the harness works
leak_importances = pd.Series(rf_leak.feature_importances_, index=FEATURES + ["ctr"]).sort_values(ascending=False)
print("\nFeature importances WITH leak:")
print(leak_importances.round(3).to_string())

# 3. Timeline check
print("\n=== Timeline check ===")
print("All 5 honest features (impressions_90d, avg_position, content_age_days, word_count,")
print("engagement_rate) are 90-day trailing aggregates or static metadata — all knowable before")
print("today's review decision. The label uses the SAME 90-day ctr snapshot, not a future window,")
print("so there is no forward-looking overlap between feature and label windows in this starter slice.")

# 4. Product-flag check
print("\n=== Product-flag check ===")
print("No FlyRank internal flags, health_score, or priority_score fields were available in this")
print("starter CSV, so none could leak in even by accident.")

Features in use: ['impressions_90d', 'avg_position', 'content_age_days', 'word_count', 'engagement_rate']
Confirmed excluded: 'ctr', 'clicks_90d' — both are directly used to build the label 'is_ctr_anomaly'

=== Leakage sanity test: reintroducing 'ctr' as a feature ===
Precision@20  |  honest features: 0.850   |   with 'ctr' reintroduced: 1.000
Precision@50  |  honest features: 0.640   |   with 'ctr' reintroduced: 0.980

Feature importances WITH leak:
ctr                 0.720
avg_position        0.130
impressions_90d     0.076
engagement_rate     0.049
word_count          0.017
content_age_days    0.007

=== Timeline check ===
All 5 honest features (impressions_90d, avg_position, content_age_days, word_count,
engagement_rate) are 90-day trailing aggregates or static metadata — all knowable before
today's review decision. The label uses the SAME 90-day ctr snapshot, not a future window,
so there is no forward-looking overlap between feature and label windows in this starter slice.

===

> **Reading the leak test:** The harness works exactly as expected. Reintroducing `ctr` as a feature pushes Precision@20
> from 0.850 to a perfect **1.000**, and Precision@50 from 0.640 to **0.980** — a near-total collapse of the ranking
> problem into trivially re-reading the label. The feature-importance breakdown confirms it directly: `ctr` alone captures
**72.0%** of total importance, dwarfing `avg_position` (13.0%), `impressions_90d` (7.6%), `engagement_rate` (4.9%),
`word_count` (1.7%), and `content_age_days` (0.7%) combined.
>
> This is the same signature Finding #1's Health Score audit showed (a handful of label-adjacent features absorbing
> almost all importance) — except here it's caused by a feature I deliberately reintroduced to prove the test itself is
> sound, not a hidden bug. Because the honest Week 5/6 features never come close to this pattern (no single feature
> exceeds ~30–40% importance in the clean run, per Week 5's error analysis), this gives confidence that the *
*Precision@50 = 0.640** grouped-split result reflects real, generalizable signal rather than an undetected leak. The gap
> between "honest" (0.640) and "leaked" (0.980) is effectively the ceiling this task could reach if leakage were present —
> and my clean model sits well below it.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

> **My boldest sentence from Week 5 (interpretation section):**
> *"The learned model is a real improvement over the hand-written rule for this ranking task..."*
>
> **Rewritten in safe language:**
> "Under a client-grouped, held-out evaluation (test clients never seen in training), the Random Forest's ranking of
> pages by predicted CTR-anomaly probability reached a **directionally observed** Precision@50 of 0.640, compared to a
> near-chance Precision@50 when the same features were tested under a naive random split that leaked client identity (
> 0.940 — inflated by 87% client overlap between train and test). This is an **observed** pattern in historical GSC/GA4
> data under one specific label definition and one specific 25-client split, not a guarantee that a 0.640 precision would
> hold on a different client mix, a different time period, or against FlyRank's actual internal flag logic. The ranked
> output is intended as **decision-support** — a way to order a human reviewer's queue — not as an automated judgment that
> a given page's metadata is broken."

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.